# Module 5 · Transects : densité et effectif par distance sampling

**Objectif :** estimer la densité et l'effectif de gorilles à partir des nids, comme le fait le logiciel *Distance*, mais en écrivant chaque étape toi-même pour comprendre ce qu'il calcule.

**Durée indicative :** 3 heures. **Prérequis :** modules 4 et 3b (index, tirages aléatoires, bootstrap).

In [ ]:
# Cellule de démarrage : exécute-la en premier (clic sur ▶ ou Maj + Entrée)
import os, sys, subprocess
if not os.path.exists("outils/verif.py"):
    if "google.colab" in sys.modules:  # Colab : on récupère les données et les vérifications
        if not os.path.exists("/content/data-foret-depot"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "claude/quirky-keller-jhyoi0",
                            "https://github.com/Cas1233128/Cr-ation_page_web", "/content/data-foret-depot"], check=True)
        os.chdir("/content/data-foret-depot/data-foret")
    else:
        raise FileNotFoundError("Le dossier 'outils' est introuvable. Ouvre ce notebook depuis le dossier data-foret (voir le README).")
from outils.verif import verifier, indice
print("Prêt. Pour chaque exercice : écris ton code, exécute, lis le verdict.")

### Point de départ (fourni)

On repart de la fiche nettoyée au module 4 et des longueurs de transects.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel("donnees/transects_propres_reference.xlsx")  # fiche nettoyée au module 4
fins = df[df["Type indice "] == "Fin"].set_index("Transec N°")["Distance transect (m)"]
L_km = fins / 1000                                                          # effort par transect (km)
strate = df.drop_duplicates("Transec N°").set_index("Transec N°")["Strate/site"].reindex(L_km.index)
superficie = df.drop_duplicates("Strate/site").set_index("Strate/site")["Sup_strate_km²"]
print(len(L_km), "transects,", L_km.sum(), "km")

### Exercice 1 · Les observations de nids

Range dans `nids` les **groupes de nids de gorille** observés sur les transects dont la longueur est connue.

In [ ]:
# À toi :
nids = 
len(nids)

verifier("m5_e1", nids)

### Exercice 2 · Les distances perpendiculaires

Range dans `x` les distances perpendiculaires **en mètres**, sans les valeurs manquantes. Puis lance l'histogramme fourni.

In [ ]:
# À toi :
x = 

plt.hist(x, bins=10, color="#2e7651")
plt.xlabel("Distance perpendiculaire (m)")
plt.ylabel("Groupes de nids")
plt.show()

verifier("m5_e2", x)

### Le principe du distance sampling

L'histogramme montre qu'on détecte beaucoup de nids près de la ligne et de moins en moins en s'éloignant : on **rate** une partie des nids éloignés. Le distance sampling modélise cette baisse par une **fonction de détection** *g(x)*, la probabilité de voir un objet situé à la distance *x*. On suppose que tout objet sur la ligne est vu (*g(0) = 1*).

Le modèle le plus courant est la **semi-normale** : $g(x) = e^{-x^2 / 2\sigma^2}$. Plus σ est grand, plus on voit loin.

On en déduit la **largeur effective de la bande (ESW)** : la largeur d'une bande où l'on verrait **tout** et qui donnerait le même nombre d'observations. La densité vient ensuite simplement : $D = \dfrac{n}{2 \, L \, ESW}$ (le 2 parce qu'on observe des deux côtés de la ligne).

### Exercice 3 · Estimer σ

Pour la semi-normale, l'estimateur du maximum de vraisemblance de σ est $\hat\sigma = \sqrt{\overline{x^2}}$ : la racine de la moyenne des distances au carré. Calcule `sigma` (en m).

In [ ]:
# À toi :
sigma = 

verifier("m5_e3", sigma)

### Exercice 4 · Largeur effective de la bande

Pour la semi-normale, $ESW = \sigma \sqrt{\pi / 2}$. Calcule `esw` (en m), puis lance le graphique fourni pour voir la courbe ajustée sur tes données.

In [ ]:
# À toi :
esw = 

grille = np.linspace(0, x.max(), 100)
plt.hist(x, bins=10, density=True, color="#cfe3d6")
plt.plot(grille, np.exp(-grille**2 / (2 * sigma**2)) / esw, color="#2e7651")
plt.xlabel("Distance (m)")
plt.show()

verifier("m5_e4", esw)

### Exercice 5 · Densité de groupes de nids

Calcule `D_groupes` en groupes par **km²**. Attention aux unités : L est en km, l'ESW en m.

In [ ]:
# À toi :
D_groupes = 

verifier("m5_e5", D_groupes)

### Exercice 6 · Densité de nids

Un groupe contient plusieurs nids (colonne `Nombre`). Calcule `D_nids` en nids/km².

In [ ]:
# À toi :
D_nids = 

verifier("m5_e6", D_nids)

### Exercice 7 · Des nids aux gorilles

Chaque gorille sevré construit environ un nid par jour (taux de production *r*), et un nid reste visible un certain temps (durée de disparition *t*). D'où :

$$D_{individus} = \frac{D_{nids}}{r \times t}$$

Utilise **r = 1.0 nid/jour** et **t = 90 jours**. Ce sont des **valeurs d'exercice** : la durée de disparition varie fortement selon le site et la saison, et devrait être mesurée localement. C'est souvent la plus grande source d'incertitude du calcul.

In [ ]:
r = 1.0
t = 90
# À toi :
D_ind = 

verifier("m5_e7", D_ind)

### Exercice 8 · Effectif par strate

La fonction de détection est commune à toutes les strates (pas assez d'observations pour l'estimer séparément), mais le taux de rencontre change. Calcule `N_strate`, le nombre de gorilles sevrés dans chaque strate : densité de la strate × superficie de la strate (`superficie`).

In [ ]:
# À toi :
N_strate = 
N_strate.round(0)

verifier("m5_e8", N_strate)

### Exercice 9 · Intervalle de confiance par bootstrap

Une estimation sans intervalle de confiance n'a pas de valeur scientifique. Le **bootstrap** estime l'incertitude en rejouant l'échantillonnage :

1. Tire **avec remise** autant de transects qu'il y en a (certains reviennent plusieurs fois, d'autres disparaissent).
2. Recalcule l'effectif total avec ces transects. Pour rester simple, garde σ et la taille moyenne des groupes fixes.
3. Répète 1000 fois. Les percentiles 2,5 % et 97,5 % des résultats forment l'intervalle de confiance à 95 %.

Calcule `N_total` (somme de `N_strate`), puis `bas` et `haut`.

> Simplification : on tire les transects dans tout l'échantillon, sans respecter les strates, et σ reste fixe. L'intervalle obtenu est donc un peu trop étroit. Le logiciel *Distance* propage toutes ces sources d'erreur.

In [ ]:
rng = np.random.default_rng(42)
N_total = 
resultats = []
# À toi :

bas, haut = 

verifier("m5_e9", bas, haut, N_total)

### Exercice 10 · Esprit critique (sans vérification)

Réponds par écrit :

1. Les guides de distance sampling recommandent **au moins 60 à 80 observations** pour estimer une fonction de détection. Combien en as-tu ? Quelle conséquence ?
2. Si la vraie durée de disparition des nids était 150 jours au lieu de 90, que deviendrait l'effectif ? Calcule-le.
3. Tu as perdu 2 nids sans distance perpendiculaire à l'exercice 2, mais tu les as comptés dans *n*. Est-ce correct ? Qu'aurait-il fallu faire ?

*Tes réponses :*

1. 
2. 
3. 

### Annexe · Le même calcul en R avec le package Distance (à lire, pas à exécuter)

C'est ce que font tes collègues avec R ou le logiciel *Distance*. Tu dois pouvoir reconnaître chaque étape.

```r
library(Distance)

# Une ligne par groupe de nids, plus une ligne par transect sans nid (distance = NA).
# Colonnes attendues : Region.Label (strate), Area (km²), Sample.Label (transect),
#                      Effort (km), distance (m), size (nids par groupe)
unites <- convert_units("meter", "kilometer", "square kilometer")

modele <- ds(donnees, key = "hn", adjustment = NULL,
             truncation = "5%", convert_units = unites)
summary(modele)   # sigma, ESW, densités par strate, IC
plot(modele)      # fonction de détection sur l'histogramme
```

Différences avec ce que tu as fait : `ds()` tronque les 5 % de distances les plus grandes, peut tester d'autres formes de fonction de détection (`key = "hr"`, ajustements) et les compare par AIC, et calcule la variance analytiquement. Le passage des nids aux individus (r et t) se fait avec des multiplicateurs (argument `multipliers` de `dht2()`, voir sa documentation).

## Bilan du module 5

Tu as reconstruit toute la chaîne : fonction de détection, ESW, densité de groupes, de nids, d'individus, effectif par strate et intervalle de confiance. Tu sais maintenant ce que le logiciel *Distance* fait, et où se cachent les principales sources d'incertitude.